<h1 style='font-weight:bold; color:#003366; text-align:center;'>Tema 1: Buenas prácticas de código (principios de Clean Code)</h1>

<h3 style='font-weight:bold; color:#003366; text-align:center;'>Programación Analítica II</h3>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>Docente Camila Palomeque</h4>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>S2 . 2026</h4>

## Objetivos del notebook

1. Entender por qué el código legible es un problema económico y no estético.
2. Aplicar los principios centrales de **Clean Code**: nombres, funciones cortas, docstrings y anotaciones de tipo.
3. Reescribir código anidado usando **cláusulas de guarda** y retornos tempranos.
4. Reconocer y eliminar repetición innecesaria (**DRY**) y números mágicos.
5. Elegir estructuras de datos eficientes y medir el impacto de esas decisiones.

**Regla mental para todo el semestre: el código se escribe una vez y se lee muchas veces. Optimizamos para la lectura.**

<h2 style='font-weight:bold; color:#003366;'>1. El costo real del código desordenado</h2>

Problemas comúnes de código:

- **No se entiende qué hace** sin ejecutarlo mentalmente línea por línea.
- **No se puede modificar** sin miedo a romper otra cosa.
- **No se puede reutilizar**, así que se copia y pega, y a partir de ahí existen n versiones y todo se vuelve aún más caótico.

<h3 style='font-weight:bold; color:#0033BB; text-align:center;'>Ejemplo 1: el mismo cálculo, dos veces</h3>

Ambas funciones devuelven exactamente el mismo resultado, fijate cuánto tiempo te lleva entender cada una.

In [1]:
# Versión A
def calc(m, t, d):
    if m > 0:
        return m + m * t - (m + m * t) * d
    return 0


print("Versión A:", calc(1000, 0.22, 0.10))

Versión A: 1098.0


In [2]:
# Versión B
def calcular_precio_final(precio_base, tasa_iva, descuento):
    """Aplica IVA y luego un descuento sobre el precio base.

    El descuento se calcula sobre el precio con impuesto incluido, tal como
    figura en la política comercial vigente.
    """
    if precio_base <= 0:
        return 0.0

    precio_con_iva = precio_base * (1 + tasa_iva)
    return precio_con_iva * (1 - descuento)


print("Versión B:", calcular_precio_final(precio_base=1000, tasa_iva=0.22, descuento=0.10))

Versión B: 1098.0


La versión B tiene más líneas y es mejor código. Esto rompe una intuición muy común, que es que menos líneas
equivale a mejor. 

**Lo que se minimiza no son las líneas, es el esfuerzo de comprensión.**

Fijate qué gana la versión B:

- El nombre dice qué hace, entonces se puede usar sin leer el cuerpo.
- El docstring (explicación entre """ """) explica la **decisión de negocio**, que es el único dato que el código no puede expresar solo.
- El paso intermedio `precio_con_iva` tiene nombre, así que el orden de las operaciones es evidente.
- La llamada usa argumentos con nombre, entonces es imposible confundir el orden de `tasa_iva` y `descuento`.

<h2 style='font-weight:bold; color:#003366;'>2. Nombres descriptivos</h2>

Elegir nombres es la actividad más frecuente al programar y la que más impacto tiene en la legibilidad.
Cuatro reglas prácticas:

- **Los nombres revelan la intención.** `dias_hasta_vencimiento` en lugar de `d`. Si el nombre necesita un comentario
al lado que lo explique, el nombre está mal elegido.

- **Los booleanos se leen como una afirmación.** `es_cliente_vip`, `tiene_mora`, `esta_vencido`. Así el `if` se lee
como una frase en español.

- **Las funciones empiezan con un verbo.** `calcular_interes`, `filtrar_activos`, `cargar_transacciones`.
Las variables son sustantivos.

- **Las colecciones van en plural.** `transacciones` es una lista, `transaccion` es un elemento. Ese detalle solo
ahorra muchísima confusión dentro de los bucles.

Nombres a evitar salvo casos muy puntuales: `x`, `data`, `temp`, `aux`, `df`, `df1`, `df2`, `resultado_final`,
`resultado_final_ok`. La excepción razonable son las variables matemáticas donde la convención es universal, por
ejemplo `x` e `y` en un gráfico o `i` como índice en un bucle corto.

<h3 style='font-weight:bold; color:#0033BB; text-align:center;'>Ejemplo 2 : renombrar un cálculo financiero</h3>

In [3]:
# Antes: hay que leer todo el cuerpo para saber qué se está calculando
def f(v, n, r):
    return v * (1 + r) ** n


print(f(10000, 5, 0.08))

14693.280768000006


In [4]:
# Después: el nombre y la firma alcanzan para entenderlo 

def valor_futuro(capital_inicial: float, anios: int, tasa_anual: float) -> float:
    """Calcula el valor futuro de un capital con interés compuesto anual."""
    return capital_inicial * (1 + tasa_anual) ** anios

print(f"Valor futuro a 5 años: {valor_futuro(capital_inicial=10_000, anios=5, tasa_anual=0.08):,.2f}")

Valor futuro a 5 años: 14,693.28


> **Tip de sintaxis:** en Python podés escribir `10_000` en lugar de `10000`. El guion bajo se ignora y hace más legibles los montos grandes sin embargo no es una práctica muy común ni obligatoria, sino meramente informativa.

También `f"{valor:,.2f}"` formatea con separador de miles y dos decimales.

<h2 style='font-weight:bold; color:#003366;'>3. Una función, una responsabilidad (código modular) </h2>

Una función debería poder describirse en una sola frase sin usar la palabra "y". Si para explicar qué hace necesitás
decir "carga el archivo **y** limpia los datos **y** calcula el total **y** lo imprime", en realidad hay cuatro
funciones escondidas ahí.

Por qué importa esto en análisis de datos:

- Podés **probar** cada paso por separado y descubrir en cuál se rompe el número.
- Podés **reutilizar** la limpieza en otro análisis sin arrastrar la impresión del reporte.
- Podés **cambiar** el criterio de una etapa sin tocar las demás.

<h3 style='font-weight:bold; color:#0033BB; text-align:center;'>Ejemplo 3: separar responsabilidades</h3>

In [5]:
# Antes: una función que hace de todo
def procesar(transacciones):
    total = 0
    for transaccion in transacciones:
        if transaccion["monto"] < 0:
            total = total + transaccion["monto"]
    print("El gasto total del período fue de:", round(abs(total), 2))
    return total


movimientos = [
    {"descripcion": "Salario", "monto": 52000},
    {"descripcion": "Alquiler", "monto": -14500},
    {"descripcion": "Supermercado", "monto": -6200},
    {"descripcion": "Freelance", "monto": 8000},
]

procesar(movimientos)

El gasto total del período fue de: 20700


-20700

In [6]:
# Después: tres piezas independientes, cada una con una responsabilidad
def filtrar_egresos(transacciones: list[dict]) -> list[dict]:
    """Devuelve solamente las transacciones con monto negativo."""
    return [transaccion for transaccion in transacciones if transaccion["monto"] < 0]


def calcular_total(transacciones: list[dict]) -> float:
    """Suma los montos de una lista de transacciones."""
    return sum(transaccion["monto"] for transaccion in transacciones)


def formatear_moneda(monto: float, simbolo: str = "$U") -> str:
    """Convierte un monto numérico en texto con separador de miles."""
    return f"{simbolo} {monto:,.2f}"


egresos = filtrar_egresos(movimientos)
gasto_total = abs(calcular_total(egresos))

print("Cantidad de egresos:", len(egresos))
print("Gasto total del período:", formatear_moneda(gasto_total))
print("Balance del período:", formatear_moneda(calcular_total(movimientos)))

Cantidad de egresos: 2
Gasto total del período: $U 20,700.00
Balance del período: $U 39,300.00


Fijate en un detalle importante: la versión limpia **no imprime nada adentro de las funciones**. Las funciones
calculan y devuelven, la impresión ocurre afuera.

Esa separación entre **calcular** y **mostrar** es una de las decisiones de diseño más valiosas que vas a tomar.

- Una función que **imprime (print)** solo sirve para mirar en pantalla.
- Una función que **devuelve (return)** sirve para graficar, exportar a Excel, encadenar con otro cálculo o testear automáticamente.

<h2 style='font-weight:bold; color:#003366;'>4. Comentarios y docstrings</h2>

La regla es corta: **el código explica el cómo, el comentario explica el por qué.**

Un comentario que repite lo que la línea ya dice es ruido, y además es peligroso, porque cuando el código cambia
el comentario queda viejo y miente.

In [7]:
# Comentario inútil: repite lo que el código ya dice
# Multiplica el monto por 0.22
impuesto = 1000 * 0.22

# Comentario útil: aporta el contexto que el código no puede expresar
TASA_IVA_BASICA = 0.22  # Tasa básica de IVA en Uruguay según DGI, vigente desde 2007
impuesto = 1000 * TASA_IVA_BASICA
print("Impuesto:", impuesto)

Impuesto: 220.0


El **docstring** es distinto de un comentario: es la documentación oficial de la función, va entre triples comillas
inmediatamente después del `def` y Python la guarda para poder consultarla.

Estructura recomendada cuando la función es no trivial:

1. Una línea que resume qué hace, en modo imperativo.
2. Línea en blanco.
3. Parámetros, qué devuelve y supuestos o casos borde relevantes.

In [8]:
def calcular_cuota_prestamo(monto, tasa_anual, plazo_meses):
    """Calcula la cuota fija mensual de un préstamo con sistema francés.

    Args:
        monto: Capital solicitado en la moneda del préstamo.
        tasa_anual: Tasa nominal anual expresada en tanto por uno, por ejemplo 0.24 para 24%.
        plazo_meses: Cantidad de cuotas mensuales.

    Returns:
        El valor de la cuota mensual constante.

    Nota:
        Si la tasa es cero la cuota es simplemente el monto dividido el plazo,
        caso que hay que tratar aparte para no dividir por cero.
    """
    tasa_mensual = tasa_anual / 12

    if tasa_mensual == 0:
        return monto / plazo_meses

    factor = (1 + tasa_mensual) ** plazo_meses
    return monto * tasa_mensual * factor / (factor - 1)


print(f"Cuota: $U {calcular_cuota_prestamo(300_000, 0.24, 36):,.2f}")

# El docstring es lo que muestra help(), por eso vale la pena escribirlo bien
help(calcular_cuota_prestamo)

Cuota: $U 11,769.86
Help on function calcular_cuota_prestamo in module __main__:

calcular_cuota_prestamo(monto, tasa_anual, plazo_meses)
    Calcula la cuota fija mensual de un préstamo con sistema francés.

    Args:
        monto: Capital solicitado en la moneda del préstamo.
        tasa_anual: Tasa nominal anual expresada en tanto por uno, por ejemplo 0.24 para 24%.
        plazo_meses: Cantidad de cuotas mensuales.

    Returns:
        El valor de la cuota mensual constante.

    Nota:
        Si la tasa es cero la cuota es simplemente el monto dividido el plazo,
        caso que hay que tratar aparte para no dividir por cero.



<h2 style='font-weight:bold; color:#003366;'>5. Anotaciones de tipo (opcionales) </h2>

Las anotaciones de tipo (o *type hints*) declaran qué tipo de dato espera cada parámetro y qué devuelve la función.

Python **no las verifica** en tiempo de ejecución, así que no son una validación. Sirven para otras tres cosas,
todas valiosas:

- **Documentan** la firma sin escribir prosa.
- El **editor autocompleta** y avisa cuando pasás algo incompatible.
- Las herramientas de análisis estático que vemos en el Tema 2 las aprovechan para detectar errores.

In [9]:
# Tipos simples
def aplicar_ajuste(monto: float, porcentaje: float) -> float:
    """Ajusta un monto por un porcentaje expresado en tanto por uno."""
    return monto * (1 + porcentaje)


# Colecciones (Python 3.9 en adelante)
def promediar(valores: list[float]) -> float:
    """Promedia una lista de valores numéricos."""
    return sum(valores) / len(valores)


def contar_por_categoria(transacciones: list[dict]) -> dict[str, int]:
    """Cuenta cuántas transacciones hay en cada categoría."""
    conteo: dict[str, int] = {}
    for transaccion in transacciones:
        categoria = transaccion["categoria"]
        conteo[categoria] = conteo.get(categoria, 0) + 1
    return conteo


# Valores que pueden faltar
def buscar_precio(catalogo: dict[str, float], producto: str) -> float | None:
    """Devuelve el precio del producto o None si no está en el catálogo."""
    return catalogo.get(producto)


datos = [
    {"categoria": "Ocio", "monto": -800},
    {"categoria": "Ocio", "monto": -1200},
    {"categoria": "Vivienda", "monto": -14500},
]
print(contar_por_categoria(datos))
print(buscar_precio({"Notebook": 42000.0}, "Monitor"))

{'Ocio': 2, 'Vivienda': 1}
None


> **Sobre `float | None`:** se lee "float o None" y se usa cuando la función puede no encontrar un resultado.
> Es mucho más honesto que devolver `0` o `-1` para señalar ausencia, porque `0` es un precio válido y esconde el error.

<h2 style='font-weight:bold; color:#003366;'>6. Estructuras de control legibles</h2>

La anidación profunda es el enemigo número uno de la legibilidad. Cada `if` dentro de otro `if` duplica los caminos
que el lector tiene que sostener en la cabeza.

Una posible técnica para mejorar el doble nivel de indentación es parametrizar los casos por fuera en un diccionario y dejar la función al nivel de if simple.

<h3 style='font-weight:bold; color:#0033BB; text-align:center;'>Ejemplo 4 : desanidar una política de descuentos</h3>

In [10]:
# Antes: cuatro caminos anidados y valores repetidos
def calcular_descuento(monto, tipo_cliente):
    if tipo_cliente == "VIP":
        if monto > 1000:
            return monto * 0.8
        else:
            return monto * 0.9
    else:
        if monto > 1000:
            return monto * 0.9
        else:
            return monto


print(calcular_descuento(1200, "VIP"), calcular_descuento(800, "Normal"))

960.0 800


In [11]:
# Después: la política queda declarada en un solo lugar y la lógica es plana
UMBRAL_COMPRA_GRANDE = 1000

DESCUENTOS = {
    ("VIP", True): 0.20,
    ("VIP", False): 0.10,
    ("Normal", True): 0.10,
    ("Normal", False): 0.00,
}


def calcular_monto_con_descuento(monto, tipo_cliente):
    """Aplica el descuento correspondiente al tipo de cliente y al tamaño de la compra."""
    es_compra_grande = monto > UMBRAL_COMPRA_GRANDE
    descuento = DESCUENTOS.get((tipo_cliente, es_compra_grande), 0.0)
    return monto * (1 - descuento)


print(calcular_monto_con_descuento(1200, "VIP"))
print(calcular_monto_con_descuento(800, "Normal"))
print(calcular_monto_con_descuento(1200, "Mayorista"))  # tipo desconocido: sin descuento

960.0
800.0
1200.0


Dos ideas potentes en esa reescritura:

**La política de negocio quedó en un diccionario, separada de la lógica.** Si mañana marketing cambia el descuento
VIP, se edita una línea de datos y no se toca la función. Esto se llama separar *configuración* de *comportamiento*.

**El caso desconocido está contemplado.** La versión original hubiera aplicado el descuento de "Normal" a un
mayorista, silenciosamente. Los errores silenciosos son los que llegan a los informes.

<h3 style='font-weight:bold; color:#0033BB; text-align:center;'>Ejemplo 5: resolver excepciones (guard clauses)</h3>

In [12]:
def evaluar_solicitud_credito(ingreso_mensual, antiguedad_anios,
                              monto_solicitado):
    """Evalúa una solicitud de crédito devolviendo el motivo del rechazo o la aprobación.

    Las validaciones se resuelven primero para que el criterio principal
    quede al final, plano y fácil de leer.
    """
    if ingreso_mensual <= 0:
        return "Rechazada: ingreso no declarado"

    if antiguedad_anios < 1:
        return "Rechazada: antigüedad laboral insuficiente"

    if monto_solicitado <= 0:
        return "Rechazada: monto inválido"

    ratio_endeudamiento = monto_solicitado / (ingreso_mensual * 12)

    if ratio_endeudamiento > 0.4:
        return "Rechazada: ratio de endeudamiento superior al 40%"

    return "Aprobada"


casos = [
    (60_000, 3, 200_000),
    (60_000, 0.5, 200_000),
    (60_000, 3, 500_000),
    (0, 5, 100_000),
]
for ingreso, antiguedad, monto in casos:
    print(f"Ingreso {ingreso:>7,} | antigüedad {antiguedad:>3} | monto {monto:>7,} -> "
          f"{evaluar_solicitud_credito(ingreso, antiguedad, monto)}")

Ingreso  60,000 | antigüedad   3 | monto 200,000 -> Aprobada
Ingreso  60,000 | antigüedad 0.5 | monto 200,000 -> Rechazada: antigüedad laboral insuficiente
Ingreso  60,000 | antigüedad   3 | monto 500,000 -> Rechazada: ratio de endeudamiento superior al 40%
Ingreso       0 | antigüedad   5 | monto 100,000 -> Rechazada: ingreso no declarado


Compará mentalmente con la alternativa: un `if` gigante con cuatro condiciones unidas por `and`, o cinco niveles de
anidación. La versión con guardas se lee de arriba hacia abajo como una lista de requisitos, y cada rechazo dice
exactamente por qué.

<h2 style='font-weight:bold; color:#003366;'>7. DRY: no te repitas</h2>

**DRY** viene de *Don't Repeat Yourself*. La idea no es evitar toda repetición de caracteres, es evitar que **una
misma decisión** viva en más de un lugar. Cuando eso pasa, tarde o temprano se actualiza una copia y no la otra.

Lo más fácil para detectarlo es darte cuenta que estás por copiar y pegar un bloque y cambiarle un valor.

En lugar de esto lo recomendables es **poner el cálculo en una función** 

In [13]:
# Antes: la fórmula del margen aparece tres veces
ventas_electronica = 850_000
costo_electronica = 620_000
ventas_hogar = 430_000
costo_hogar = 310_000
ventas_libros = 95_000
costo_libros = 61_000

margen_electronica = (ventas_electronica - costo_electronica) / ventas_electronica * 100
margen_hogar = (ventas_hogar - costo_hogar) / ventas_hogar * 100
margen_libros = (ventas_libros - costo_libros) / ventas_libros * 100

print(round(margen_electronica, 1), round(margen_hogar, 1), round(margen_libros, 1))

27.1 27.9 35.8


In [14]:
# Después: la fórmula existe una sola vez y los datos son datos
def calcular_margen_porcentual(ventas, costo):
    if ventas == 0:
        return 0.0
    return (ventas - costo) / ventas * 100


resultados_por_categoria = {
    "Electrónica": (850_000, 620_000),
    "Hogar": (430_000, 310_000),
    "Libros": (95_000, 61_000),
}

for categoria, (ventas, costo) in resultados_por_categoria.items():
    print(f"{categoria:<14} margen: {calcular_margen_porcentual(ventas, costo):5.1f} %")

Electrónica    margen:  27.1 %
Hogar          margen:  27.9 %
Libros         margen:  35.8 %


Observá el beneficio oculto: agregar la categoría "Deportes" ahora es agregar **una línea de datos**, no tres líneas
de código. Y si el criterio de margen cambia a margen sobre costo, se corrige en un único lugar.

<h2 style='font-weight:bold; color:#003366;'>8. Números mágicos y constantes con nombre</h2>

Un **número mágico** es un valor literal metido en medio del código sin explicación: `0.22`, `1000`, `0.4`, `365`.

Quien lee no sabe si `0.22` es el IVA, una tasa de descuento o una probabilidad. Y si ese mismo valor aparece en
cinco lugares, actualizarlo es una búsqueda arqueológica.

La convención en Python es definir constantes en **MAYÚSCULAS** al comienzo del archivo. Python no impide
modificarlas, pero la convención comunica con total claridad que no hay que hacerlo.

In [15]:
# Constantes del módulo: se definen una vez, arriba, y se leen solas
TASA_IVA_BASICA = 0.22
DIAS_HABILES_ANIO = 250
RATIO_ENDEUDAMIENTO_MAXIMO = 0.40
UMBRAL_CLIENTE_PREMIUM = 500_000


def precio_con_iva(precio_sin_impuesto):
    return precio_sin_impuesto * (1 + TASA_IVA_BASICA)


def es_cliente_premium(facturacion_anual):
    return facturacion_anual >= UMBRAL_CLIENTE_PREMIUM


print(f"Precio con IVA: $U {precio_con_iva(1_000):,.2f}")
print("¿Es premium con 620.000 de facturación?:", es_cliente_premium(620_000))

Precio con IVA: $U 1,220.00
¿Es premium con 620.000 de facturación?: True


<h2 style='font-weight:bold; color:#003366;'>9. Estructuras de datos y costo computacional</h2>

Elegir bien la estructura de datos suele tener más impacto en el rendimiento que cualquier microoptimización.
La intuición central es cómo busca cada estructura:

| Estructura | Sintaxis | Búsqueda de un elemento | Cuándo usarla |
|---|---|---|---|
| `list` | `[1, 2, 3]` | Recorre todo, costo proporcional al tamaño | Secuencia ordenada que se recorre completa |
| `tuple` | `(1, 2, 3)` | Igual que la lista | Datos que no deben cambiar, por ejemplo una coordenada o una fila |
| `set` | `{1, 2, 3}` | Directa, costo casi constante | Preguntar pertenencia o eliminar duplicados |
| `dict` | `{"a": 1}` | Directa por clave | Asociar una clave con un valor, por ejemplo un id de cliente con sus datos |

La diferencia entre recorrer y acceder directo es imperceptible con 10 elementos y brutal con 100.000.

In [16]:
import time

# Simulamos una cartera de 200.000 identificadores de cliente
cantidad_clientes = 200_000
ids_lista = [f"CL{numero:06d}" for numero in range(cantidad_clientes)]
ids_conjunto = set(ids_lista)

id_buscado = "CL199999"  # el peor caso para la lista: está al final

inicio = time.perf_counter()
for _ in range(100):
    encontrado_en_lista = id_buscado in ids_lista
tiempo_lista = time.perf_counter() - inicio

inicio = time.perf_counter()
for _ in range(100):
    encontrado_en_conjunto = id_buscado in ids_conjunto
tiempo_conjunto = time.perf_counter() - inicio

print(f"Búsqueda en lista: {tiempo_lista:.5f} segundos")
print(f"Búsqueda en set: {tiempo_conjunto:.5f} segundos")
print(f"El conjunto fue aproximadamente {tiempo_lista / max(tiempo_conjunto, 1e-9):,.0f} veces más rápido")

Búsqueda en lista: 0.28383 segundos
Búsqueda en set: 0.00007 segundos
El conjunto fue aproximadamente 3,992 veces más rápido


Cuidado con la conclusión: **no siempre conviene un `set`**. Un set no conserva el orden y no admite duplicados,
que a veces son exactamente lo que necesitás. 

La regla útil es: **si tu código pregunta `in` dentro de un bucle, convertí la colección a `set` antes del bucle**.

In [17]:
# Patrón muy frecuente y muy costoso: preguntar "in" sobre una lista dentro de un bucle
clientes_morosos = [f"CL{numero:06d}" for numero in range(0, 200_000, 3)]
cartera_a_revisar = [f"CL{numero:06d}" for numero in range(0, 4_000)]

# Versión lenta
inicio = time.perf_counter()
morosos_encontrados = [cliente for cliente in cartera_a_revisar if cliente in clientes_morosos]
tiempo_lento = time.perf_counter() - inicio

# Versión rápida: una sola conversión antes del bucle
inicio = time.perf_counter()
morosos_set = set(clientes_morosos)
morosos_encontrados_rapido = [cliente for cliente in cartera_a_revisar if cliente in morosos_set]
tiempo_rapido = time.perf_counter() - inicio

print(f"Con lista: {tiempo_lento:.4f} s")
print(f"Con set: {tiempo_rapido:.4f} s")
print("Mismo resultado:", morosos_encontrados == morosos_encontrados_rapido)

Con lista: 2.1189 s
Con set: 0.0043 s
Mismo resultado: True


<h2 style='font-weight:bold; color:#003366;'>10. Comprensiones de listas</h2>

Una **comprensión de lista** construye una lista describiendo su contenido en una sola expresión. La estructura es
siempre la misma:

```python
[expresion for elemento in coleccion if condicion]
```

Se lee de izquierda a derecha como una frase: "la expresión, para cada elemento de la colección, si cumple la condición".

Ventajas frente al `for` con `append`: es más corta, más rápida y deja claro de
un vistazo que el resultado es una lista nueva.

Y una advertencia igual de importante: **una comprensión ilegible es peor que un `for` claro**. Si necesitás dos
condiciones, un `if/else` y un cálculo intermedio, escribí el bucle.

In [18]:
transacciones = [
    {"descripcion": "Salario", "monto": 52_000, "categoria": "Ingresos"},
    {"descripcion": "Alquiler", "monto": -14_500, "categoria": "Vivienda"},
    {"descripcion": "Tienda Inglesa", "monto": -6_200, "categoria": "Supermercado"},
    {"descripcion": "Netflix", "monto": -380, "categoria": "Ocio"},
    {"descripcion": "Freelance", "monto": 8_000, "categoria": "Ingresos"},
]

# Filtrar
egresos = [t for t in transacciones if t["monto"] < 0]

# Transformar
descripciones = [t["descripcion"].upper() for t in transacciones]

# Filtrar y transformar a la vez
montos_egresos_absolutos = [abs(t["monto"]) for t in transacciones if t["monto"] < 0]

# Con condicional en la expresión
etiquetas = ["Ingreso" if t["monto"] > 0 else "Egreso" for t in transacciones]

print("Egresos:", len(egresos))
print("Descripciones:", descripciones[:3])
print("Montos de egreso:", montos_egresos_absolutos)
print("Etiquetas:", etiquetas)

Egresos: 3
Descripciones: ['SALARIO', 'ALQUILER', 'TIENDA INGLESA']
Montos de egreso: [14500, 6200, 380]
Etiquetas: ['Ingreso', 'Egreso', 'Egreso', 'Egreso', 'Ingreso']


In [19]:
# También existen comprensiones de diccionario y de conjunto
gasto_por_descripcion = {t["descripcion"]: abs(t["monto"]) for t in transacciones if t["monto"] < 0}
categorias_unicas = {t["categoria"] for t in transacciones}

print(gasto_por_descripcion)
print(categorias_unicas)

# Y expresiones generadoras, que no construyen la lista en memoria.
# Cuando solo vas a sumar o contar, esto ahorra memoria: notá la ausencia de corchetes.
total_egresos = sum(abs(t["monto"]) for t in transacciones if t["monto"] < 0)
print("Total de egresos:", total_egresos)

{'Alquiler': 14500, 'Tienda Inglesa': 6200, 'Netflix': 380}
{'Ingresos', 'Vivienda', 'Supermercado', 'Ocio'}
Total de egresos: 21080


> **Cuándo NO usar una comprensión:** cuando dentro del bucle hacés varias cosas, cuando necesitás un `try/except`,
> o cuando la línea resultante no entra cómoda en la pantalla. La legibilidad gana siempre.

<h2 style='font-weight:bold; color:#003366;'>11. PEP 8 : la guía de estilo de Python</h2>

**PEP 8** es el documento oficial que define cómo se escribe Python. No es una obligación técnica, es la convención
que hace que todo el código Python del mundo se parezca y por lo tanto se lea rápido.

Lo mínimo que hay que respetar en este curso:

| Regla | Correcto | Incorrecto |
|---|---|---|
| Indentación de 4 espacios | `    return x` | `  return x` |
| Funciones y variables en `guión bajo` | `calcular_total` | `CalcularTotal`, `calcularTotal` |
| Clases en `PascalCase` (opcional)| `class CarteraCliente:` | `class cartera_cliente:` |
| Constantes en `MAYUSCULAS` (opcional pero útil) | `TASA_IVA = 0.22` | `tasaIva = 0.22` |
| Espacios alrededor de operadores | `total = a + b` | `total=a+b` |
| Sin espacio antes del paréntesis | `sumar(a, b)` | `sumar (a, b)` |
| Máximo 79 a 88 caracteres por línea | línea corta | línea de 200 caracteres |
| Dos líneas en blanco entre funciones | `\n\ndef otra():` | funciones pegadas |
| Imports arriba, uno por línea | `import numpy as np` | `import numpy, pandas` |

La buena noticia es que **nada de esto se memoriza**: en el Tema 2 instalamos herramientas que lo verifican y
lo corrigen automáticamente. Lo que sí conviene entender es el criterio detrás.

In [20]:
# Este bloque viola varias reglas de PEP 8 a propósito.

def CalcularTotal( lista, IVA = 0.22 ):
    Total=0
    for i in range(len( lista )):
        Total=Total+lista[i]*(1+IVA)
    return Total
print (CalcularTotal([100,200,300]))

732.0


In [21]:
# Código "limpio":

def calcular_total(lista, iva=0.22):
    total = 0
    for i in range(len(lista)):
        total = total + lista[i] * (1 + iva)
    return total
print(calcular_total([100, 200, 300]))

732.0


<h2 style='font-weight:bold; color:#003366;'>12. Checklist de código limpio</h2>

Usá esta lista antes de entregar cualquier práctico del curso:

- [ ] Los nombres de variables y funciones dicen qué contienen o qué hacen, sin abreviaturas.
- [ ] Cada función se puede describir en una frase sin usar "y" o si tienen varios pasos los mismos tienen sentido.
- [ ] Las funciones devuelven valores, no imprimen resultados intermedios.
- [ ] Toda función no explica tiene docstring.
- [ ] No hay números mágicos: los valores fijos son constantes con nombre.
- [ ] No hay más de dos niveles de anidación.
- [ ] No hay bloques copiados y pegados con cambios menores.
- [ ] No quedaron variables ni imports sin usar.

<h2 style='font-weight:bold; color:#003366; text-align:center;'>Ejercicios para resolver en clase (con solución)</h2>

La cantidad de asteriscos indica la dificultad. Se recomienda:

1. Leer el enunciado y pensar el problema antes de escribir código.
2. Resolverlo en la celda vacía que sigue.
3. Recién después desplegar la **solución oculta** para comparar los enfoques.

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 1 *</h3>


Refactorizá la siguiente función aplicando **al menos dos** principios vistos: nombres descriptivos, declaración de tipo, docstring. No cambies el comportamiento.

In [22]:
def proc(l):
    r = []
    for i in range(len(l)):
        if l[i] % 2 == 0:
            r.append(l[i] * 2)
    return r


print(proc([1, 2, 3, 4, 5, 6]))

[4, 8, 12]


In [23]:
# Tu versión limpia acá

<details>
<summary><b>Ver solución</b></summary>

```python
def duplicar_numeros_pares(numeros: list[int]) -> list[int]:
    """Devuelve una lista con los números pares de la entrada, multiplicados por dos."""
    return [numero * 2 for numero in numeros if numero % 2 == 0]


print(duplicar_numeros_pares([1, 2, 3, 4, 5, 6]))
```

</details>

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 2 *</h3>


La función de abajo calcula el precio final de un producto. Tiene tres problemas: reasigna la misma variable,
usa números mágicos en la llamada y no documenta el orden de las operaciones.

Reescribila con constantes con nombre, pasos intermedios nombrados y docstring que aclare si el descuento se
aplica antes o después del impuesto.

In [24]:
def calcular_precio_final(precio_base, impuesto, descuento):
    precio = precio_base + (precio_base * impuesto)
    precio = precio - (precio * descuento)
    return precio


print(calcular_precio_final(100, 0.22, 0.10))

109.8


In [25]:
# Tu versión limpia acá

<details>
<summary><b>Ver solución</b></summary>

```python
TASA_IVA_BASICA = 0.22


def calcular_precio_final(precio_base,
                          tasa_impuesto = TASA_IVA_BASICA,
                          descuento = 0.0) -> float:
    """Calcula el precio final aplicando primero el impuesto y después el descuento.

    El orden importa: el descuento se calcula sobre el precio con impuesto incluido,
    que es el criterio comercial que usamos en el curso.
    """
    precio_con_impuesto = precio_base * (1 + tasa_impuesto)
    return precio_con_impuesto * (1 - descuento)


print(f"$U {calcular_precio_final(100, descuento=0.10):,.2f}")
```

</details>

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 3 **</h3>


La siguiente función clasifica una nota. Funciona, pero tiene la escala de calificación repartida dentro de la lógica.

Reescribila de modo que **la escala esté declarada como dato** en una estructura aparte y la función solo recorra
esa estructura. Así, cambiar la escala no implica tocar la función.

In [26]:
def clasificar(nota):
    if nota >= 90:
        return "A"
    elif nota >= 80:
        return "B"
    elif nota >= 70:
        return "C"
    elif nota >= 60:
        return "D"
    else:
        return "F"


print(clasificar(83))

B


In [27]:
# Tu versión limpia acá

<details>
<summary><b>Ver solución</b></summary>

```python
# La escala es configuración, no lógica: ordenada de mayor a menor
ESCALA_CALIFICACION = [
    (90, "A"),
    (80, "B"),
    (70, "C"),
    (60, "D"),
]
CALIFICACION_INSUFICIENTE = "F"


def clasificar_nota(nota: float) -> str:
    """Devuelve la letra correspondiente a la nota según la escala vigente."""
    for nota_minima, letra in ESCALA_CALIFICACION:
        if nota >= nota_minima:
            return letra
    return CALIFICACION_INSUFICIENTE


for nota_ejemplo in (95, 83, 71, 64, 40):
    print(nota_ejemplo, "->", clasificar_nota(nota_ejemplo))
```

</details>

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 4 ***</h3>


**Ejercicio integrador.** El siguiente bloque calcula indicadores de una cartera de préstamos. Está escrito de la
peor manera posible: nombres opacos, números mágicos, anidación, repetición y una búsqueda ineficiente dentro de
un bucle.

Refactorizalo por completo. El resultado numérico debe ser idéntico.

In [28]:
p = [
    {"id": "CL001", "m": 300000, "t": 0.24, "pl": 36, "s": "Privado"},
    {"id": "CL002", "m": 150000, "t": 0.28, "pl": 24, "s": "Independiente"},
    {"id": "CL003", "m": 800000, "t": 0.19, "pl": 60, "s": "Público"},
    {"id": "CL004", "m": 220000, "t": 0.26, "pl": 12, "s": "Privado"},
    {"id": "CL005", "m": 95000, "t": 0.31, "pl": 18, "s": "Independiente"},
]
mor = ["CL002", "CL005"]

tot = 0
totm = 0
c = 0
cm = 0
for i in range(len(p)):
    tm = p[i]["t"] / 12
    f = (1 + tm) ** p[i]["pl"]
    cu = p[i]["m"] * tm * f / (f - 1)
    tot = tot + cu
    c = c + 1
    if p[i]["id"] in mor:
        totm = totm + cu
        cm = cm + 1

print("Cuota promedio:", round(tot / c, 2))
print("Cuota promedio morosos:", round(totm / cm, 2))
print("Tasa de mora:", round(cm / c * 100, 1), "%")

Cuota promedio: 13687.7
Cuota promedio morosos: 7449.79
Tasa de mora: 40.0 %


In [29]:
# Tu versión limpia acá

<details>
<summary><b>Ver solución</b></summary>

```python
MESES_POR_ANIO = 12


def calcular_cuota_francesa(monto: float, tasa_anual: float, plazo_meses: int) -> float:
    """Calcula la cuota mensual constante de un préstamo con sistema francés."""
    tasa_mensual = tasa_anual / MESES_POR_ANIO
    if tasa_mensual == 0:
        return monto / plazo_meses
    factor = (1 + tasa_mensual) ** plazo_meses
    return monto * tasa_mensual * factor / (factor - 1)


def calcular_indicadores_cartera(prestamos: list[dict], ids_morosos: set[str]) -> dict[str, float]:
    """Calcula cuota promedio de la cartera, de los morosos y la tasa de mora.

    Recibe los morosos como conjunto para que la verificación de pertenencia
    sea directa y no recorra una lista en cada iteración.
    """
    cuotas = [calcular_cuota_francesa(p["monto"], p["tasa_anual"], p["plazo_meses"])
              for p in prestamos]
    cuotas_morosos = [cuota for p, cuota in zip(prestamos, cuotas)
                      if p["id_cliente"] in ids_morosos]

    return {
        "cuota_promedio": sum(cuotas) / len(cuotas),
        "cuota_promedio_morosos": sum(cuotas_morosos) / len(cuotas_morosos),
        "tasa_mora_porcentual": len(cuotas_morosos) / len(cuotas) * 100,
    }


prestamos = [
    {"id_cliente": "CL001", "monto": 300_000, "tasa_anual": 0.24, "plazo_meses": 36, "sector": "Privado"},
    {"id_cliente": "CL002", "monto": 150_000, "tasa_anual": 0.28, "plazo_meses": 24, "sector": "Independiente"},
    {"id_cliente": "CL003", "monto": 800_000, "tasa_anual": 0.19, "plazo_meses": 60, "sector": "Público"},
    {"id_cliente": "CL004", "monto": 220_000, "tasa_anual": 0.26, "plazo_meses": 12, "sector": "Privado"},
    {"id_cliente": "CL005", "monto": 95_000, "tasa_anual": 0.31, "plazo_meses": 18, "sector": "Independiente"},
]
IDS_MOROSOS = {"CL002", "CL005"}

indicadores = calcular_indicadores_cartera(prestamos, IDS_MOROSOS)
print(f"Cuota promedio          : $U {indicadores['cuota_promedio']:,.2f}")
print(f"Cuota promedio morosos  : $U {indicadores['cuota_promedio_morosos']:,.2f}")
print(f"Tasa de mora            : {indicadores['tasa_mora_porcentual']:.1f} %")
```

</details>